# 1.0 Method #

In [1]:
import numpy as np
import pandas as pd
import time
from codecarbon import EmissionsTracker, OfflineEmissionsTracker
import re
from transformers import BertModel, BertTokenizer
import torch
from scipy.stats import skew
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from tabulate import tabulate
import matplotlib.pyplot as plt
import torch.nn.functional as F

[codecarbon WARNING @ 16:36:39] Nvidia GPU detected but pynvml initialization failed. Please ensure NVIDIA drivers are properly installed.


In [2]:
TEXT_COL = "sentence"
LABEL_COL = "label"

SAMPLE_SIZE = 200
RANDOM_SEED = 42
BATCH_SIZE = 64
MAX_TOKEN = 128

MODELS = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased"
]

### 1.1 Negative Correlation Based Asymmetry Testing

In [3]:
def asymmetry_eta(x, bandwidth=None):
    """
    Estimate eta using Patil et al.-style estimator:
    eta_hat = -Corr(f_hat(X_i), F_hat(X_i))

    f_hat(X_i): leave-one-out Gaussian KDE
    F_hat(X_i): leave-one-out empirical CDF
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)

    if n < 3:
        return np.nan

    # Silverman's bandwidth
    s = np.std(x, ddof=1)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    sigma = min(s, iqr / 1.349) if iqr > 0 else s
    if np.isclose(sigma, 0):
        sigma = s if s > 0 else 1.0
    h = bandwidth if bandwidth is not None else 0.9 * sigma * (n ** (-1 / 5))#calculate bandwidth
    if h <= 0:
        h = 1.06 * (s if s > 0 else 1.0) * (n ** (-1 / 5))

    # Leave-one-out KDE
    diffs = (x[None, :] - x[:, None]) / h#cal culate pairwise difference
    K = stats.norm.pdf(diffs)#fit standardized distance into gaussian kernel
    np.fill_diagonal(K, 0.0)#leave one out
    f_hat = K.sum(axis=1) / ((n - 1) * h)#KED for pdf

    # Leave-one-out empirical CDF
    indicators = (x[None, :] < x[:, None]).astype(float)
    np.fill_diagonal(indicators, 0.0)
    F_hat = indicators.sum(axis=1) / (n - 1)

    if np.std(f_hat, ddof=1) == 0 or np.std(F_hat, ddof=1) == 0:
        return np.nan

    eta = -np.corrcoef(f_hat, F_hat)[0, 1]#calculate correlation

    return eta

### 1.2 Distribution distance based differentiation testing

In [4]:
def chi2_distance(dist1, dist2, bins=50, value_range=None, eps=1e-12):

    hist1, bin_edges = np.histogram(dist1, bins=bins, range=value_range)
    hist2, _ = np.histogram(dist2, bins=bin_edges)

    p = hist1 / hist1.sum()
    q = hist2 / hist2.sum()

    chi_dist = 0.5 * np.sum((p - q) ** 2 / (p + q + eps))

    return chi_dist

### 1.2 All pairwise cosine similarity

In [5]:
def calculate_pairwise_cosine_distribution(embeddings):
    scores = []

    for i in range(len(embeddings)):
        for j in range(i + 1, len(embeddings)):
            sim = torch.nn.functional.cosine_similarity(
                embeddings[i],
                embeddings[j],
                dim=0
            )
            scores.append(sim.item())

    return np.array(scores)

### 1.3 Pairwise Cosine simimarity (same label pairs)

In [6]:
def calculate_within_label_cosine_distribution(
    embeddings,
    labels,
    return_by_label=True
):

    labels = np.asarray(labels)
    unique_labels = np.unique(labels)

    all_scores = []#saving all results from different labels
    scores_by_label = {}#dictionary, saving results for each label

    for lab in unique_labels:
        idx = np.where(labels == lab)[0]

        # Need at least 2 samples to calculate pairwise cosine
        if len(idx) < 2:
            print(f"Label {lab}: skipped, only {len(idx)} sample")
            continue

        scores = []

        # Calculate cosine similarity only inside the same label
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                i = idx[a]
                j = idx[b]

                sim = torch.nn.functional.cosine_similarity(
                    embeddings[i],
                    embeddings[j],
                    dim=0
                )

                scores.append(sim.item())

        scores = np.array(scores)

        scores_by_label[lab] = scores
        all_scores.extend(scores)

        print(f"Label {lab}: {len(idx)} samples, {len(scores)} pairs")

    all_scores = np.array(all_scores)

    if return_by_label:
        return all_scores, scores_by_label
    else:
        return all_scores

### 1.4 Pairwise Cosine Similarity (Same and Different label)

In [7]:
def calculate_between_label_cosine_distribution(
    embeddings,
    labels,
    return_by_label_pair=True
):
    labels = np.asarray(labels)
    
    all_scores = []
    scores_by_label_pair = {}

    n = len(labels)

    for i in range(n):
        for j in range(i + 1, n):

            # only calculate pairs from different labels
            if labels[i] == labels[j]:
                continue

            lab_i = labels[i]
            lab_j = labels[j]

            sim = torch.nn.functional.cosine_similarity(
                embeddings[i],
                embeddings[j],
                dim=0
            )

            score = sim.item()
            all_scores.append(score)

            # make label pair order fixed, e.g. (0, 2), not sometimes (2, 0)
            label_pair = tuple(sorted((lab_i, lab_j)))

            if label_pair not in scores_by_label_pair:
                scores_by_label_pair[label_pair] = []

            scores_by_label_pair[label_pair].append(score)

    # convert to numpy arrays
    all_scores = np.array(all_scores)

    for pair in scores_by_label_pair:
        scores_by_label_pair[pair] = np.array(scores_by_label_pair[pair])

    print("Total between-label pairs:", len(all_scores))

    for pair, scores in scores_by_label_pair.items():
        print(f"Label {pair[0]} vs Label {pair[1]}: {len(scores)} pairs")

    if return_by_label_pair:
        return all_scores, scores_by_label_pair
    else:
        return all_scores

## 2.0 Dataset

In [8]:
csv = "financial_original.csv"
df = pd.read_csv(csv)
print(df.head())
print(df.columns)
print(df.shape)

                                            sentence  label
0  According to Gran , the company has no plans t...      1
1  Technopolis plans to develop in stages an area...      1
2  The international electronic industry company ...      0
3  With the new production plant the company woul...      2
4  According to the company 's updated strategy f...      2
Index(['sentence', 'label'], dtype='str')
(4846, 2)


In [9]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"[?|!|\'|\"|#]", "", text)
    text = re.sub(r"[.|,|)|(|\\|/]", " ", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_and_clean_dataset(csv_file):
    df = pd.read_csv(csv_file)
    df = df[[TEXT_COL, LABEL_COL]].dropna().drop_duplicates().copy()

    df["clean_sentence"] = df[TEXT_COL].apply(clean_text)
    df["length"] = df["clean_sentence"].apply(len)

    print(df.head())
    print(df[LABEL_COL].value_counts())
    print(df.shape)

    return df

In [10]:
def add_length_quantile_groups(df, num_quantiles=100):
    df = df.copy()

    quantiles = df["length"].quantile(
        [i / num_quantiles for i in range(1, num_quantiles)]
    ).tolist()

    quantiles = np.unique(quantiles).tolist()

    bins = [df["length"].min()] + quantiles + [df["length"].max()]
    labels = [f"Q{i+1}" for i in range(len(bins) - 1)]

    df["quantiles"] = pd.cut(
        df["length"],
        bins=bins,
        labels=labels,
        include_lowest=True,
        duplicates="drop"
    )

    print(df["quantiles"].value_counts().sort_index())

    return df

In [11]:
df = load_and_clean_dataset(csv)

                                            sentence  label  \
0  According to Gran , the company has no plans t...      1   
1  Technopolis plans to develop in stages an area...      1   
2  The international electronic industry company ...      0   
3  With the new production plant the company woul...      2   
4  According to the company 's updated strategy f...      2   

                                      clean_sentence  length  
0  According to Gran the company has no plans to ...     121  
1  Technopolis plans to develop in stages an area...     186  
2  The international electronic industry company ...     224  
3  With the new production plant the company woul...     204  
4  According to the company s updated strategy fo...     198  
label
1    2873
2    1363
0     604
Name: count, dtype: int64
(4840, 4)


In [12]:
df = add_length_quantile_groups(df)
print(df.head())
print(df.columns)
print(df.shape)

quantiles
Q1      51
Q2      46
Q3      55
Q4      43
Q5      47
        ..
Q96     41
Q97     49
Q98     50
Q99     48
Q100    46
Name: count, Length: 100, dtype: int64
                                            sentence  label  \
0  According to Gran , the company has no plans t...      1   
1  Technopolis plans to develop in stages an area...      1   
2  The international electronic industry company ...      0   
3  With the new production plant the company woul...      2   
4  According to the company 's updated strategy f...      2   

                                      clean_sentence  length quantiles  
0  According to Gran the company has no plans to ...     121       Q54  
1  Technopolis plans to develop in stages an area...     186       Q86  
2  The international electronic industry company ...     224       Q95  
3  With the new production plant the company woul...     204       Q91  
4  According to the company s updated strategy fo...     198       Q89  
Index(['sente

## 3.0 Data Sampling

In [13]:
def sample_from_length_groups(df, sample_size=200, num_quantiles=100, random_seed=42):
    n_samples = sample_size // num_quantiles

    sampled_df = (
        df.groupby("quantiles", group_keys=False)
        .apply(lambda x: x.sample(n=min(n_samples, len(x)), random_state=random_seed))
        .reset_index(drop=True)
    )

    print("Actual Sample Size:", len(sampled_df))
    print("Minimum Token Count:", sampled_df["clean_sentence"].str.split().apply(len).min())
    print("Average Token Count:", sampled_df["clean_sentence"].str.split().apply(len).mean())
    print("Maximum Token Count:", sampled_df["clean_sentence"].str.split().apply(len).max())

    return sampled_df

## 3.0 Embedding

In [14]:
def compute_embeddings(texts, model_name, batch_size=64, max_token=128):
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_token
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu()

        embeddings.append(batch_embeddings)

    return torch.cat(embeddings)

## 4.0 Cosine Similarity Resampling

基于bootstrap的resampling,用same label distribution里的生成的label pair数量作为参考，使不同label互相pair 也能形成同样的样本数量。正常pair下，不同label的pair数量会比相同label pair少很多，因此在sample size 不同下无法对比双方cosine similarity的分布。因此，从每一种（不同组合的diff） label pair中抽取相同数量的pair 使其相加后对等于相同label的pair,并加在一起形成分布。但是由于随机抽取的随机性，无法使用仅抽取一次的分布代表整体diff label 分布与，也无法直接对比两种分布，所以用bootstrap,重复抽取1000次（最好计算好cosine similarity后再抽取cosine similarity即可）。每次都计算需要的数值：分布距离，然后取平均值（expected value）（或者中位数等），作为当前两种分布的对比结果。

In [15]:
def bootstrap_chi2_from_existing_scores(
    same_scores,
    diff_label,
    n_bootstrap=1000,
    random_seed=42,
    bins=50,
    value_range=(-1, 1),
    eta_bandwidth=None
):

    rng = np.random.default_rng(random_seed)

    same_scores = np.asarray(same_scores)
    target_size = len(same_scores)

    # same-label statistics, fixed across bootstrap
    same_mean = same_scores.mean()
    same_eta = asymmetry_eta(same_scores, bandwidth=eta_bandwidth)
    same_abs_eta = abs(same_eta)

    label_pairs = list(diff_label.keys())
    n_pair_types = len(label_pairs)

    base_n = target_size // n_pair_types
    remainder = target_size % n_pair_types

    bootstrap_rows = []

    print("Same-label target size:", target_size)
    print("Different-label pair types:", n_pair_types)
    print("Base sample per diff-label pair:", base_n)

    for b in range(n_bootstrap):
        sampled_diff_scores = []

        for k, pair in enumerate(label_pairs):
            # already-computed cosine similarity scores for this label pair
            cosine_scores_for_this_pair = np.asarray(diff_label[pair])

            n_sample = base_n + (1 if k < remainder else 0)

            sampled_cosine_scores = rng.choice(
                cosine_scores_for_this_pair,
                size=n_sample,
                replace=True
            )

            sampled_diff_scores.extend(sampled_cosine_scores)

        sampled_diff_scores = np.asarray(sampled_diff_scores)

        # chi-square distance between same-label and bootstrapped diff-label distributions
        chi2_dist = chi2_distance(
            dist1=same_scores,
            dist2=sampled_diff_scores,
            bins=bins,
            value_range=value_range
        )

        # bootstrap diff-label statistics
        diff_mean = sampled_diff_scores.mean()
        diff_eta = asymmetry_eta(sampled_diff_scores, bandwidth=eta_bandwidth)
        diff_abs_eta = abs(diff_eta)

        mean_gap = same_mean - diff_mean
        abs_mean_gap = abs(mean_gap)

        eta_gap = same_eta - diff_eta
        abs_eta_gap = abs(eta_gap)

        chi2_result = {
            "bootstrap": b + 1,

            # same-label statistics, fixed
            "same_mean": same_mean,
            "same_eta": same_eta,
            "same_abs_eta": same_abs_eta,

            # bootstrapped diff-label statistics
            "diff_mean": diff_mean,
            "diff_eta": diff_eta,
            "diff_abs_eta": diff_abs_eta,

            # gaps
            "mean_gap": mean_gap,
            "abs_mean_gap": abs_mean_gap,
            "eta_gap": eta_gap,
            "abs_eta_gap": abs_eta_gap,

            # distribution distance
            "chi2_distance": chi2_dist,

            # pair numbers
            "same_n_pairs": len(same_scores),
            "diff_n_pairs": len(sampled_diff_scores)
        }

        bootstrap_rows.append(chi2_result)

    bootstrap_chi2_df = pd.DataFrame(bootstrap_rows)

    chi2_summary = {
        # same-label statistics
        "same_mean": same_mean,
        "same_eta": same_eta,
        "same_abs_eta": same_abs_eta,

        # averaged bootstrapped diff-label statistics
        "diff_mean": bootstrap_chi2_df["diff_mean"].mean(),
        "diff_mean_std": bootstrap_chi2_df["diff_mean"].std(),

        "diff_eta": bootstrap_chi2_df["diff_eta"].mean(),
        "diff_eta_std": bootstrap_chi2_df["diff_eta"].std(),
        "diff_abs_eta": bootstrap_chi2_df["diff_abs_eta"].mean(),

        # averaged gaps
        "mean_gap": bootstrap_chi2_df["mean_gap"].mean(),
        "mean_gap_std": bootstrap_chi2_df["mean_gap"].std(),
        "abs_mean_gap": bootstrap_chi2_df["abs_mean_gap"].mean(),

        "eta_gap": bootstrap_chi2_df["eta_gap"].mean(),
        "eta_gap_std": bootstrap_chi2_df["eta_gap"].std(),
        "abs_eta_gap": bootstrap_chi2_df["abs_eta_gap"].mean(),

        # averaged chi-square distance
        "chi2_distance": bootstrap_chi2_df["chi2_distance"].mean(),
        "chi2_distance_std": bootstrap_chi2_df["chi2_distance"].std(),

        # metadata
        "same_n_pairs": len(same_scores),
        "diff_n_pairs": bootstrap_chi2_df["diff_n_pairs"].mean(),
        "n_bootstrap": n_bootstrap,
        "bins": bins,
        "value_range": value_range
    }

    return bootstrap_chi2_df, chi2_summary

## 5.0 Running Test

for different pairing method, the running method is different

In [16]:
def run_one_model_chi2_bootstrap(
    sampled_df,
    model_name,
    batch_size=64,
    max_token=128,
    label_col="label",
    n_bootstrap=1000,
    random_seed=42,
    bins=50,
    value_range=(-1, 1),
    eta_bandwidth=None,
):
    print(f"\nRunning model: {model_name}")

    texts = sampled_df["clean_sentence"].tolist()
    labels = sampled_df[label_col].values

    # 1. Compute embeddings only once
    embeddings = compute_embeddings(
        texts,
        model_name,
        batch_size=batch_size,
        max_token=max_token
    )

    # 2. Same-label cosine distribution
    same_scores, scores_by_label = calculate_within_label_cosine_distribution(
        embeddings=embeddings,
        labels=labels,
        return_by_label=True
    )

    same_eta = asymmetry_eta(same_scores)

    # 3. Different-label cosine distribution
    # This step computes all different-label cosine similarities only once.
    diff_scores_original, diff_label = calculate_between_label_cosine_distribution(
        embeddings=embeddings,
        labels=labels,
        return_by_label_pair=True
    )


    # 5. Bootstrap different-label distribution
    # Important: this function only samples from existing diff_label[pair].
    # It does NOT recompute cosine similarity.
    bootstrap_chi2_df, chi2_summary = bootstrap_chi2_from_existing_scores(
        same_scores=same_scores,
        diff_label=diff_label,
        n_bootstrap=n_bootstrap,
        random_seed=random_seed,
        bins=bins,
        value_range=value_range,
        eta_bandwidth=eta_bandwidth
    )

    summary = {
        "model": model_name,

        # same-label
        "same_mean": np.mean(same_scores),
        "same_eta": same_eta,
        "same_abs_eta": abs(same_eta),
        "same_n_pairs": len(same_scores),

        # bootstrapped different-label
        "diff_mean": chi2_summary["diff_mean"],
        "diff_n_pairs": chi2_summary["diff_n_pairs"],
        "mean_gap": chi2_summary["mean_gap"],
        "abs_mean_gap": chi2_summary["abs_mean_gap"],

        # chi-square distance
        "chi2_distance": chi2_summary["chi2_distance"],
        "chi2_distance_std": chi2_summary["chi2_distance_std"],

        "n_bootstrap": n_bootstrap,
        "bins": bins,
        "value_range": value_range
    }

    print("\nSummary:")
    print("Same mean:", summary["same_mean"])
    print("Diff mean:", summary["diff_mean"])
    print("Mean gap:", summary["mean_gap"])
    print("Chi2 distance:", summary["chi2_distance"])
    print("Chi2 distance std:", summary["chi2_distance_std"])
    print("Same eta:", summary["same_eta"])

    return {
        "summary": summary,

        # already-computed distributions
        "same_scores": same_scores,
        "diff_scores_original": diff_scores_original,

        # same-label and diff-label grouped scores
        "scores_by_label": scores_by_label,
        "diff_label": diff_label,

        # bootstrap result
        "bootstrap_chi2_df": bootstrap_chi2_df,

        # embeddings and labels
        "embeddings": embeddings,
        "labels": labels
    }

In [17]:
def running_sizes_chi2_bootstrap(
    grouped_df,
    sample_size=200,
    num_quantiles=100,
    num_repetitions=3,
    models=MODELS,
    batch_size=64,
    max_token=128,
    label_col="label",
    n_bootstrap=1000,
    bins=50,
    value_range=(-1, 1),
    eta_bandwidth=None
):
    summary_results = []
    full_results = {}

    for repetition in range(num_repetitions):
        print("=================================================")
        print(f"Repetition {repetition + 1}/{num_repetitions}")

        random_seed = np.random.randint(1000)

        sampled_df = sample_from_length_groups(
            grouped_df,
            sample_size=sample_size,
            num_quantiles=num_quantiles,
            random_seed=random_seed
        )

        print("Actual Sample Size:", len(sampled_df))
        print("Label counts:")
        print(sampled_df[label_col].value_counts())

        for model_name in models:
            result = run_one_model_chi2_bootstrap(
                sampled_df=sampled_df,
                model_name=model_name,
                batch_size=batch_size,
                max_token=max_token,
                label_col=label_col,
                n_bootstrap=n_bootstrap,
                random_seed=random_seed,
                bins=bins,
                value_range=value_range,
                eta_bandwidth=eta_bandwidth
            )

            summary = result["summary"].copy()

            summary["repetition"] = repetition + 1
            summary["random_seed"] = random_seed
            summary["sample_size"] = sample_size

            summary_results.append(summary)

            full_results[(repetition + 1, model_name)] = result

    summary_df = pd.DataFrame(summary_results)

    return {
        "summary_df": summary_df,
        "full_results": full_results
    }

In [18]:
model_order = [
    "recobo/agriculture-bert-uncased",
    "google-bert/bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "allenai/scibert_scivocab_uncased",
    "Lianglab/PharmBERT-uncased",
    "recobo/chemical-bert-uncased",
    "nlpaueb/legal-bert-base-uncased"
]


In [19]:
chi2_results = running_sizes_chi2_bootstrap(
    grouped_df=df,
    sample_size=200,
    num_quantiles=100,
    num_repetitions=3,
    models=model_order,
    batch_size=64,
    max_token=128,
    label_col="label",
    n_bootstrap=1000,
    bins=50,
    value_range=(-1, 1),
    eta_bandwidth=None,
)

chi2_summary_df = chi2_results["summary_df"]
chi2_summary_df

Repetition 1/3
Actual Sample Size: 200
Minimum Token Count: 5
Average Token Count: 21.39
Maximum Token Count: 72
Actual Sample Size: 200
Label counts:
label
1    133
2     43
0     24
Name: count, dtype: int64

Running model: recobo/agriculture-bert-uncased


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: recobo/agriculture-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

Label 0: 24 samples, 276 pairs
Label 1: 133 samples, 8778 pairs
Label 2: 43 samples, 903 pairs
Total between-label pairs: 9943
Label 0 vs Label 1: 3192 pairs
Label 1 vs Label 2: 5719 pairs
Label 0 vs Label 2: 1032 pairs
Same-label target size: 9957
Different-label pair types: 3
Base sample per diff-label pair: 3319

Summary:
Same mean: 0.8228108187187289
Diff mean: 0.8369863458443348
Mean gap: -0.014175527125606031
Chi2 distance: 0.02866220041555799
Chi2 distance std: 0.0014643999064032252
Same eta: -0.25964105139477295

Running model: google-bert/bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (55 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label 0: 35 samples, 595 pairs
Label 1: 120 samples, 7140 pairs
Label 2: 45 samples, 990 pairs
Total between-label pairs: 11175
Label 1 vs Label 2: 5400 pairs
Label 0 vs Label 1: 4200 pairs
Label 0 vs Label 2: 1575 pairs
Same-label target size: 8725
Different-label pair types: 3
Base sample per diff-label pair: 2908

Summary:
Same mean: 0.8708960448705023
Diff mean: 0.8769247677452927
Mean gap: -0.006028722874790314
Chi2 distance: 0.008100720782178843
Chi2 distance std: 0.000885444393003397
Same eta: -0.15129288250786213


,model,same_mean,same_eta,same_abs_eta,same_n_pairs,diff_mean,diff_n_pairs,mean_gap,abs_mean_gap,chi2_distance,chi2_distance_std,n_bootstrap,bins,value_range,repetition,random_seed,sample_size
0,recobo/agriculture-bert-uncased,0.822811,-0.259641,0.259641,9957,0.836986,9957.0,-0.014176,0.014176,0.028662,0.001464,1000,50,"(-1, 1)",1,493,200
1,google-bert/bert-base-uncased,0.670201,-0.004421,0.004421,9957,0.695120,9957.0,-0.024919,0.024919,0.027982,0.001621,1000,50,"(-1, 1)",1,493,200
2,ahmedrachid/FinancialBERT,0.749979,-0.260182,0.260182,9957,0.753900,9957.0,-0.003921,0.003921,0.007456,0.000704,1000,50,"(-1, 1)",1,493,200
3,allenai/scibert_scivocab_uncased,0.755673,-0.346255,0.346255,9957,0.770866,9957.0,-0.015193,0.015193,0.017023,0.001182,1000,50,"(-1, 1)",1,493,200
4,Lianglab/PharmBERT-uncased,0.738830,-0.292047,0.292047,9957,0.751538,9957.0,-0.012709,0.012709,0.009645,0.000950,1000,50,"(-1, 1)",1,493,200
5,recobo/chemical-bert-uncased,0.817034,-0.461803,0.461803,9957,0.826869,9957.0,-0.009835,0.009835,0.010309,0.000890,1000,50,"(-1, 1)",1,493,200
6,nlpaueb/legal-bert-base-uncased,0.869166,-0.263069,0.263069,9957,0.877620,9957.0,-0.008454,0.008454,0.018134,0.001176,1000,50,"(-1, 1)",1,493,200
7,recobo/agriculture-bert-uncased,0.826658,-0.289067,0.289067,9532,0.831187,9532.0,-0.004530,0.004530,0.003540,0.000561,1000,50,"(-1, 1)",2,33,200
8,google-bert/bert-base-uncased,0.681097,-0.107421,0.107421,9532,0.696593,9532.0,-0.015495,0.015495,0.011615,0.001041,1000,50,"(-1, 1)",2,33,200
9,ahmedrachid/FinancialBERT,0.751244,-0.124073,0.124073,9532,0.747517,9532.0,0.003728,0.003728,0.003261,0.000530,1000,50,"(-1, 1)",2,33,200


In [20]:
chi2_summary_df = chi2_summary_df.sort_values(
    "chi2_distance",
    ascending=False
)

chi2_summary_df

,model,same_mean,same_eta,same_abs_eta,same_n_pairs,diff_mean,diff_n_pairs,mean_gap,abs_mean_gap,chi2_distance,chi2_distance_std,n_bootstrap,bins,value_range,repetition,random_seed,sample_size
0,recobo/agriculture-bert-uncased,0.822811,-0.259641,0.259641,9957,0.836986,9957.0,-0.014176,0.014176,0.028662,0.001464,1000,50,"(-1, 1)",1,493,200
1,google-bert/bert-base-uncased,0.670201,-0.004421,0.004421,9957,0.695120,9957.0,-0.024919,0.024919,0.027982,0.001621,1000,50,"(-1, 1)",1,493,200
6,nlpaueb/legal-bert-base-uncased,0.869166,-0.263069,0.263069,9957,0.877620,9957.0,-0.008454,0.008454,0.018134,0.001176,1000,50,"(-1, 1)",1,493,200
3,allenai/scibert_scivocab_uncased,0.755673,-0.346255,0.346255,9957,0.770866,9957.0,-0.015193,0.015193,0.017023,0.001182,1000,50,"(-1, 1)",1,493,200
18,Lianglab/PharmBERT-uncased,0.741586,-0.082121,0.082121,8725,0.756753,8725.0,-0.015167,0.015167,0.013404,0.001105,1000,50,"(-1, 1)",3,927,200
15,google-bert/bert-base-uncased,0.687449,0.072331,0.072331,8725,0.703498,8725.0,-0.016049,0.016049,0.012317,0.001064,1000,50,"(-1, 1)",3,927,200
8,google-bert/bert-base-uncased,0.681097,-0.107421,0.107421,9532,0.696593,9532.0,-0.015495,0.015495,0.011615,0.001041,1000,50,"(-1, 1)",2,33,200
19,recobo/chemical-bert-uncased,0.813298,-0.378275,0.378275,8725,0.826713,8725.0,-0.013415,0.013415,0.011089,0.001004,1000,50,"(-1, 1)",3,927,200
11,Lianglab/PharmBERT-uncased,0.731230,-0.299732,0.299732,9532,0.745562,9532.0,-0.014332,0.014332,0.010990,0.000978,1000,50,"(-1, 1)",2,33,200
17,allenai/scibert_scivocab_uncased,0.757176,-0.306963,0.306963,8725,0.769319,8725.0,-0.012143,0.012143,0.010913,0.000990,1000,50,"(-1, 1)",3,927,200


In [26]:
# 1. Calculate mean chi2 distance across repetitions for each model
chi2_summary_df

,model,chi2_mean,chi2_std,n_repetitions
0,Lianglab/PharmBERT-uncased,0.011346,0.001905,3
1,ahmedrachid/FinancialBERT,0.005995,0.002370,3
2,allenai/scibert_scivocab_uncased,0.011368,0.005442,3
3,google-bert/bert-base-uncased,0.017305,0.009254,3
4,nlpaueb/legal-bert-base-uncased,0.008947,0.008795,3
5,recobo/agriculture-bert-uncased,0.013432,0.013385,3
6,recobo/chemical-bert-uncased,0.009756,0.001679,3


In [28]:
chi2_ordered_df = (
    chi2_summary_df
    .sort_values(by="chi2_mean", ascending=False)
    .reset_index(drop=True)
)

chi2_ordered_df

,model,chi2_mean,chi2_std,n_repetitions
0,google-bert/bert-base-uncased,0.017305,0.009254,3
1,recobo/agriculture-bert-uncased,0.013432,0.013385,3
2,allenai/scibert_scivocab_uncased,0.011368,0.005442,3
3,Lianglab/PharmBERT-uncased,0.011346,0.001905,3
4,recobo/chemical-bert-uncased,0.009756,0.001679,3
5,nlpaueb/legal-bert-base-uncased,0.008947,0.008795,3
6,ahmedrachid/FinancialBERT,0.005995,0.002370,3


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


def cluster_models_by_ks(results_df):
    results_df = results_df.copy()

    data = results_df[
        [
            "ks_distance",
            "same_abs_eta"
        ]
    ].values

    data_scaled = StandardScaler().fit_transform(data)

    labels = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    ).fit_predict(data_scaled)

    results_df["cluster"] = labels

    cluster_score = results_df.groupby("cluster").agg(
        ks_score=("ks_distance", "mean"),
        eta_score=("same_abs_eta", "mean")
    )

    # Normalize cluster-level score
    ks_min = cluster_score["ks_score"].min()
    ks_max = cluster_score["ks_score"].max()

    eta_min = cluster_score["eta_score"].min()
    eta_max = cluster_score["eta_score"].max()

    if ks_max != ks_min:
        cluster_score["ks_norm"] = (
            cluster_score["ks_score"] - ks_min
        ) / (ks_max - ks_min)
    else:
        cluster_score["ks_norm"] = 0.0

    if eta_max != eta_min:
        cluster_score["eta_norm"] = (
            cluster_score["eta_score"] - eta_min
        ) / (eta_max - eta_min)
    else:
        cluster_score["eta_norm"] = 0.0

    # larger KS is better, smaller eta is better
    cluster_score["final_score"] = (
        cluster_score["ks_norm"]
        - cluster_score["eta_norm"]
    )

    ordered_clusters = cluster_score["final_score"].sort_values(
        ascending=False
    ).index.tolist()

    cluster_name_map = {
        ordered_clusters[0]: "high-fit",
        ordered_clusters[1]: "less-fit"
    }

    results_df["fit_group"] = results_df["cluster"].map(cluster_name_map)

    return results_df

In [ ]:
clustered_ks_df = cluster_models_by_ks(ks_summary_df)
clustered_ks_df

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))

for group in ["high-fit", "less-fit"]:
    sub = clustered_ks_df[clustered_ks_df["fit_group"] == group]

    if sub.empty:
        continue

    plt.scatter(
        sub["ks_distance"],
        sub["same_abs_eta"],
        label=group
    )

    for _, row in sub.iterrows():
        name = row["model"].split("/")[-1]

        plt.annotate(
            name,
            (row["ks_distance"], row["same_abs_eta"]),
            fontsize=8
        )

plt.xlabel("KS Distance between Same-label and Different-label Distributions")
plt.ylabel("Absolute Same-label Eta")
plt.title("Model Clustering Based on KS Distance and Eta")
plt.legend()
plt.tight_layout()
plt.show()

Model Performance Ranking （In paper)

| Rank | Model | Accuracy / Micro F1 | Macro F1 | LMDFit Group |
|---:|---|---:|---:|---|
| 1 | AgricultureBERT | 0.844 | 0.830 | more-fit |
| 2 | BERT | 0.840 | 0.825 | more-fit |
| 3 | FinancialBERT | 0.841 | 0.824 | more-fit |
| 4 | SciBERT | 0.840 | 0.823 | less-fit |
| 5 | PharmBERT | 0.835 | 0.810 | more-fit |
| 6 | ChemicalBERT | 0.729 | 0.592 | less-fit |
| 7 | LegalBERT | 0.731 | 0.576 | less-fit |

## Observation: Global Cosine Similarity Distribution and Model Fitness

From the cosine similarity distribution plots, the best downstream model does not necessarily produce the most Gaussian-like or most symmetric global distribution.

BERT produces the most symmetric distribution, with Eta close to 0 and a relatively moderate mean cosine similarity. Its downstream performance is also strong, which partly supports the assumption that a more symmetric similarity distribution may indicate better model fitness.

However, AgricultureBERT achieves the best Macro F1 score, even though its cosine similarity distribution is not the most Gaussian-like. Its distribution is more concentrated and shifted to the right, with a higher mean cosine similarity. This suggests that global distributional symmetry alone is not sufficient to explain downstream classification performance.

In contrast, LegalBERT and ChemicalBERT show distributions that are highly concentrated toward high cosine similarity values. This means that many text pairs are represented as overly similar in the embedding space. Their downstream Macro F1 scores are also relatively low, which supports the paper's assumption that a cosine similarity distribution with a mean closer to 1 may indicate weaker discriminative ability.

Overall, the results suggest that Eta can be useful as an asymmetry descriptor, but it should not be used alone as a complete fitness indicator. Model fitness for text classification depends not only on the global shape of the cosine similarity distribution, but also on whether the model embeddings can separate different classes effectively.

Therefore, global Eta partially reflects model fitness, but it has limitations. A possible future improvement is to move from global cosine similarity distribution analysis to class-aware analysis, such as comparing intra-class and inter-class cosine similarity distributions.

## 5.0 Model Clustering

Mean cosine similarity measures the average closeness among all text embeddings generated by a model.

If the mean is too high, many texts are mapped very close to each other in the embedding space. This suggests that the model may not differentiate text samples well, making downstream classification harder.

Therefore, following the paper's assumption, a higher mean cosine similarity is treated as a penalty for model fitness.

<b> Higher mean value have to be punished more, but it's hard to decide the punishment, based on financial dataset, the best model architecture it actually have a relatively high mean value.</b>

In [ ]:
def cluster_models_eta_2groups(results_df):
    results_df = results_df.copy()

    data = results_df[["eta", "mean"]].values

    labels = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    ).fit_predict(data)

    results_df["cluster"] = labels

    # Calculate average absolute eta for each cluster
    cluster_eta_score = results_df.groupby("cluster")["eta"].mean().abs()

    # Sort clusters: closest to 0 = high-fit
    ordered_clusters = cluster_eta_score.sort_values().index.tolist()

    cluster_name_map = {
        ordered_clusters[0]: "high-fit",
        ordered_clusters[1]: "less-fit"
    }

    results_df["fit_group"] = results_df["cluster"].map(cluster_name_map)

    return results_df

In [ ]:
clustered_df_eta = cluster_models_eta_2groups(eta_summary_df)
clustered_df_eta

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))

for group in ["high-fit", "less-fit"]:
    sub = clustered_df_eta[clustered_df_eta["fit_group"] == group]

    if sub.empty:
        continue

    plt.scatter(
        sub["eta"],
        sub["mean"],
        label=group
    )

    for _, row in sub.iterrows():
        name = row["model"].split("/")[-1]

        plt.annotate(
            name,
            (row["eta"], row["mean"]),
            fontsize=8
        )

plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel("Eta")
plt.ylabel("Mean Cosine Similarity")
plt.title("Model Clustering Based on Overall Within-label Eta and Mean")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
valid_model_order = [
    model for model in model_order
    if model in clustered_df_eta["model"].values
]

ordered_eta_table = (
    clustered_df_eta
    .set_index("model")
    .loc[valid_model_order]
    .reset_index()
)

ordered_eta_table = ordered_eta_table[
    ["model", "mean", "eta", "abs_eta", "fit_group"]
]

ordered_eta_table